# 7 · Analyse enhancer redundancy (local)

Reads the per-comparison redundancy tables from notebook 6 and the DESeq2 pairwise results. Computes the **frequency** of redundancy per comparison and compares **|log2FC|** between redundancy genes and non-redundant switchers (hypothesis: redundancy genes have smaller |log2FC| — conserved expression).

In [ ]:
import os.path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

REDUNDANCY_DIR = "../../data/whole_chromosomes/enhancer_redundancy"
DESEQ_DIR = "../../data/deseq"
FIGS = "figs"
EXPORT = "export"

# comparison -> (deseq parquet, flip_log2fc?) — align sign to c1->c2 direction.
# notebook 3 stores gm12878_vs_h1esc, h1esc_vs_hffc6, gm12878_vs_hffc6;
# reversed directions flip the sign relative to the stored file.
DESEQ = {
    "gm12878_vs_h1esc": ("gm12878_vs_h1esc_results.parquet", False),
    "h1esc_vs_gm12878": ("gm12878_vs_h1esc_results.parquet", True),
    "h1esc_vs_hffc6":   ("h1esc_vs_hffc6_results.parquet", False),
    "hffc6_vs_h1esc":   ("h1esc_vs_hffc6_results.parquet", True),
    "gm12878_vs_hffc6": ("gm12878_vs_hffc6_results.parquet", False),
    "hffc6_vs_gm12878": ("gm12878_vs_hffc6_results.parquet", True),
}

def redundancy_frequency(df):
    denom = int(df['nearest_changed'].sum())
    return float('nan') if denom == 0 else int(df['is_redundancy'].sum()) / denom

def compare_abs_log2fc(redundancy_lfc, contrast_lfc):
    # Mann-Whitney U on |log2FC|; rank-biserial r > 0 => redundancy smaller
    a = np.abs(np.asarray(redundancy_lfc, dtype=float)); a = a[~np.isnan(a)]
    b = np.abs(np.asarray(contrast_lfc, dtype=float)); b = b[~np.isnan(b)]
    n1, n2 = len(a), len(b)
    if n1 == 0 or n2 == 0:
        return {'n_redundancy': n1, 'n_contrast': n2, 'median_redundancy': np.nan,
                'median_contrast': np.nan, 'U': np.nan, 'pvalue': np.nan, 'rank_biserial': np.nan}
    res = stats.mannwhitneyu(a, b, alternative='two-sided')
    return {'n_redundancy': n1, 'n_contrast': n2,
            'median_redundancy': float(np.median(a)), 'median_contrast': float(np.median(b)),
            'U': float(res.statistic), 'pvalue': float(res.pvalue),
            'rank_biserial': 1.0 - (2.0 * res.statistic) / (n1 * n2)}

In [ ]:
def load_comparison(comparison):
    rt = pd.read_parquet(os.path.join(REDUNDANCY_DIR, comparison))
    fname, flip = DESEQ[comparison]
    # DESeq2 parquet stores the gene id as the index (named 'Geneid' here) with a
    # version suffix (ENSG....N); bring it to a 'gene_id' column and strip versions.
    de = pd.read_parquet(os.path.join(DESEQ_DIR, fname)).reset_index()
    gid = next((c for c in ['gene_id', 'Geneid', 'index'] if c in de.columns), de.columns[0])
    de = de.rename(columns={gid: 'gene_id'})[['gene_id', 'log2FoldChange', 'padj']].copy()
    if flip:
        de['log2FoldChange'] = -de['log2FoldChange']
    de['gene_id'] = de['gene_id'].astype(str).str.split('.').str[0]
    rt['gene_id'] = rt['gene_id'].astype(str).str.split('.').str[0]
    m = rt.merge(de, on='gene_id', how='inner')
    m['abs_log2fc'] = m['log2FoldChange'].abs()
    return m

In [ ]:
rows = []
joined = {}
for comparison in DESEQ:
    try:
        m = load_comparison(comparison)
    except FileNotFoundError:
        print(f"skip {comparison} (no parquet yet)")
        continue
    joined[comparison] = m
    rows.append({'comparison': comparison, 'n_genes': len(m),
                 'n_changed': int(m.nearest_changed.sum()),
                 'n_redundancy': int(m.is_redundancy.sum()),
                 'frequency': redundancy_frequency(m)})
freq_df = pd.DataFrame(rows)
freq_df.to_csv(os.path.join(EXPORT, "enhancer_redundancy_frequency.csv"), index=False)

if not freq_df.empty:
    ax = freq_df.plot.bar(x='comparison', y='frequency', legend=False, figsize=(8, 4))
    ax.set_ylabel("redundancy frequency")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGS, "exp6_frequency.png"), dpi=150)
    plt.show()
freq_df

In [ ]:
stat_rows = []
for comparison, m in joined.items():
    red = m.loc[m.is_redundancy, 'abs_log2fc'].values
    con = m.loc[m.is_nonredundant_switcher, 'abs_log2fc'].values
    s = compare_abs_log2fc(red, con)
    s['comparison'] = comparison
    stat_rows.append(s)
stats_df = pd.DataFrame(stat_rows)
if not stats_df.empty:
    stats_df = stats_df[['comparison', 'n_redundancy', 'n_contrast', 'median_redundancy',
                         'median_contrast', 'U', 'pvalue', 'rank_biserial']]
    stats_df.to_csv(os.path.join(EXPORT, "enhancer_redundancy_log2fc_stats.csv"), index=False)
stats_df

In [ ]:
for comparison, m in joined.items():
    sub = m.assign(group=np.where(m.is_redundancy, 'redundancy',
                          np.where(m.is_nonredundant_switcher, 'non_redundant', None)))
    sub = sub.dropna(subset=['group'])
    if sub.empty:
        continue
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    for g, gg in sub.groupby('group'):
        xs = np.sort(gg['abs_log2fc'].values)
        a1.plot(xs, np.linspace(0, 1, len(xs)), label=g)
    a1.set_xlabel("|log2FC|"); a1.set_ylabel("ECDF"); a1.legend(); a1.set_title(comparison)
    sns.violinplot(data=sub, x='group', y='abs_log2fc', ax=a2, cut=0)
    a2.set_title("|log2FC| by group")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGS, f"exp6_log2fc_{comparison}.png"), dpi=150)
    plt.show()